# **Discretization Schemes for Monte Carlo Methods**


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

Assume Black-Scholes framework and closed-form Black-Scholes formula as benchmark.




In [ ]:
def bs_formula(S_0, K, sigma, r, T, option_type="call"):
  d1=(np.log(S_0/K)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
  d2=d1-sigma*np.sqrt(T)
  if option_type=="call":
    price=S_0*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
  elif option_type=="put":
    price=K*np.exp(-r*T)*norm.cdf(-d2)-S_0*norm.cdf(-d1)
  else:
    raise ValueError ("option_type should be 'call' or 'put'")

  return price

The Euler scheme provides the simplest first-order approximation of the stochastic differential equation:

$$
S_{i+1}^{E}=S_i^{E}+rS_i^{E}\Delta t+\sigma S_i^{E}\sqrt{\Delta t}\, Z_i.
$$


The Milstein scheme adds a correction term that improves the approximation of the diffusion component:

$$
S_{i+1}^{M}=S_i^{M}+rS_i^{M}\Delta t+\sigma S_i^{M}\sqrt{\Delta t}\, Z_i
+\frac{1}{2}\sigma^2 S_i^{M}\left((\sqrt{\Delta t} Z_i)^2-\Delta t\right).
$$

In [ ]:
def path_simulation(S_0, sigma, r, T, N, n):
  """
  Simulate the terminal asset price using the exact solution, the Euler scheme,
  and the Milstein scheme on the same Brownian increments.
  """
    Z = np.random.standard_normal((N, n))
    dt = T / n

    S_true = np.full(N, S_0, dtype=float)
    S_euler = np.full(N, S_0, dtype=float)
    S_milstein = np.full(N, S_0, dtype=float)

    for i in range(n):
        Zi = Z[:, i]
        S_true = S_true * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Zi)
        S_euler = S_euler + r * S_euler * dt + sigma * S_euler * np.sqrt(dt) * Zi
        S_milstein = S_milstein + r * S_milstein * dt + sigma * S_milstein * np.sqrt(dt) * Zi + 0.5 * sigma**2 * S_milstein * ((np.sqrt(dt) * Zi)**2 - dt)

    return S_true, S_euler, S_milstein

In [ ]:
def mc_price(S_T, K, r, T, option_type="call"):
  if option_type=="call":
    payoff=np.maximum(S_T-K,0)
  elif option_type=="put":
    payoff=np.maximum(K-S_T,0)
  else:
    raise ValueError("option_type should be 'call' or 'put'")
  discounted_payoff=payoff*np.exp(-r*T)

  return np.mean(discounted_payoff)

We compare the schemes in terms of strong error and weak error.

The strong error measures how close the simulated paths are to the exact solution:
$$
e_{\mathrm{strong}}=\left(\mathbb{E}\left[|S_T-\widehat{S}_T|^2\right]\right)^{1/2}.
$$


In [ ]:
def strong_error(S_true, S_approx):
  return np.sqrt(np.mean((S_true-S_approx)**2))

The weak error measures the impact of discretization on option pricing:
$$
e_{\mathrm{weak}}=\left|\mathbb{E}[f(S_T)]-\mathbb{E}[f(\widehat{S}_T)]\right|.
$$

In [ ]:
def weak_error(S_true, S_approx, K, r, T, option_type="call"):
  exact_price=mc_price(S_true, K, r, T, option_type)
  approx_price=mc_price(S_approx, K, r, T, option_type)
  return np.abs(exact_price-approx_price)

# Numerical example

In [ ]:
S_0=100
K=100
sigma=0.2
r=0.02
T=2
N=100000
n=500

In [ ]:
S_true, S_euler, S_milstein =path_simulation(S_0, sigma, r, T, N, n)


In [ ]:
price_eu=mc_price(S_euler, K, r, T, option_type="call")
strong_err_eu=strong_error(S_true, S_euler)
weak_err_eu=weak_error(S_true, S_euler, K,r,T, option_type="call")

print(" ## EULER DISCRETIZATION ## ")
print("Price:", price_eu)
print("Strong error:", strong_err_eu)
print("Weak error:", weak_err_eu)

 ## EULER DISCRETIZATION ## 
Price: 13.054601054364436
Strong error: 0.2749237978303891
Weak error: 0.0007826433408997246


In [ ]:
price_mil=mc_price(S_milstein, K, r, T, option_type="call")
strong_err_mil=strong_error(S_true, S_milstein)
weak_err_mil=weak_error(S_true, S_milstein, K,r,T, option_type="call")

print(" ## MILSTEIN DISCRETIZATION ## ")
print("Price:", price_mil)
print("Strong error:", strong_err_mil)
print("Weak error:", weak_err_mil)

 ## MILSTEIN DISCRETIZATION ## 
Price: 13.054421114432165
Strong error: 0.00327451747375649
Weak error: 0.0009625832731714468
